###**تثبيت المكتبات**

In [2]:
!pip install -q flask pyngrok

### **استيراد المكتبات**

In [3]:
from flask import Flask, request, jsonify
import sqlite3
from datetime import datetime
import threading

### **Webhooks انشاء قاعدة بيانات وجدول تسجيل ال**

In [4]:
def init_database():
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS webhook_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            received_data TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)

    connection.commit()
    connection.close()


init_database()

print("قاعدة البيانات جاهزة")

قاعدة البيانات جاهزة


### `بيفتح قاعدة بيانات او بيعملها لو مش موجودة`
sqlite3.connect("store.db")


### `لو مش موجود اصلا webhook_log  بيقول لبايثون اعمل جدول `
CREATE TABLE IF NOT EXISTS webhook_logs
`
### `id والجدول فيه `

### `رقم Request لكل`
received_data

### `ال وصلتله JSON ال `
timestamp

### `Flask Serverانشاء `

In [5]:
app = Flask(__name__)

print("Flask app جاهز")

Flask app جاهز


### `Endpoint نعمل  `

In [6]:
@app.route("/webhook", methods=["POST"])
def webhook():
    data = request.get_json(silent=True)

    if data is None:
        return jsonify({
            "status": "error",
            "message": "Invalid or missing JSON"
        }), 400

    print("البيانات الواردة:")
    print(data)

    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute(
        """
        INSERT INTO webhook_logs (received_data, timestamp)
        VALUES (?, ?)
        """,
        (str(data), datetime.now().isoformat())
    )

    connection.commit()
    connection.close()

    return jsonify({
        "status": "received"
    }), 200

### `تشغيل السيرفر`

In [7]:
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)


server_thread = threading.Thread(target=run_flask)
server_thread.daemon = True
server_thread.start()

print("Flask server بدأ على Port 5000")

Flask server بدأ على Port 5000


## `إنشاء Public URL لاختبار الـ Webhook`

In [19]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3JlGPjlhDGHm53UM4YXtpp3qmRu_592MErEdkeDYtjQqJLZZM"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5000)

print("Public URL:")
print(public_url)

Public URL:
NgrokTunnel: "https://kilowatt-yoga-sixties.ngrok-free.dev" -> "http://localhost:5000"


In [21]:
webhook_url = public_url.public_url + "/webhook"

print("Webhook URL:")
print(webhook_url)

Webhook URL:
https://kilowatt-yoga-sixties.ngrok-free.dev/webhook


### `Webhook اختبار   `

In [22]:
import requests

test_data = {
    "name": "Aya",
    "event": "new_member",
    "score": 95
}

response = requests.post(
    webhook_url,
    json=test_data
)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 06:02:47] "POST /webhook HTTP/1.1" 200 -


البيانات الواردة:
{'name': 'Aya', 'event': 'new_member', 'score': 95}
Status Code: 200
Response: {'status': 'received'}


### `بتأكد ان البيانات اتخزنت فعلا في قاعدة البيانات`

In [23]:
import sqlite3

connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    SELECT id, received_data, timestamp
    FROM webhook_logs
""")

logs = cursor.fetchall()

connection.close()

for log in logs:
    print(log)

(1, "{'name': 'Aya', 'event': 'new_member', 'score': 95}", '2026-09-24T06:02:47.487162')


### `بتأكد ان كل ريكويست جديد بيتضاف كـ سجل جديد`

In [24]:
test_data_2 = {
    "name": "Phantoms",
    "event": "test_webhook",
    "message": "Hello from W2D3"
}

response = requests.post(
    webhook_url,
    json=test_data_2
)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 06:06:58] "POST /webhook HTTP/1.1" 200 -


البيانات الواردة:
{'name': 'Phantoms', 'event': 'test_webhook', 'message': 'Hello from W2D3'}
Status Code: 200
Response: {'status': 'received'}


### `بثبت هنا ان الامرين اتحفظوا`

In [25]:
import sqlite3

connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    SELECT id, received_data, timestamp
    FROM webhook_logs
    ORDER BY id
""")

logs = cursor.fetchall()

connection.close()

for log in logs:
    print(log)

(1, "{'name': 'Aya', 'event': 'new_member', 'score': 95}", '2026-09-24T06:02:47.487162')
(2, "{'name': 'Phantoms', 'event': 'test_webhook', 'message': 'Hello from W2D3'}", '2026-09-24T06:06:58.026092')


### `التاسك طالب ان ال ويب هوك يقبل بوست بس فهبعتله جيت ونشوف الفلاسك هيرفض ولا لأ`

In [26]:
response = requests.get(webhook_url)

print("Status Code:", response.status_code)
print("Response:", response.text)

INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 06:09:47] "GET /webhook HTTP/1.1" 405 -


Status Code: 405
Response: <!doctype html>
<html lang=en>
<title>405 Method Not Allowed</title>
<h1>Method Not Allowed</h1>
<p>The method is not allowed for the requested URL.</p>



In [27]:
%%writefile W2D3_Webhook_Receiver.py

from flask import Flask, request, jsonify
import sqlite3
import json
from datetime import datetime

app = Flask(__name__)


def init_database():
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS webhook_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            received_data TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)

    connection.commit()
    connection.close()


@app.route("/webhook", methods=["POST"])
def webhook():
    data = request.get_json(silent=True)

    if data is None:
        return jsonify({
            "status": "error",
            "message": "Invalid or missing JSON"
        }), 400

    print("البيانات الواردة:")
    print(data)

    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute(
        """
        INSERT INTO webhook_logs (received_data, timestamp)
        VALUES (?, ?)
        """,
        (
            json.dumps(data, ensure_ascii=False),
            datetime.now().isoformat()
        )
    )

    connection.commit()
    connection.close()

    return jsonify({
        "status": "received"
    }), 200


if __name__ == "__main__":
    init_database()

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False
    )

Writing W2D3_Webhook_Receiver.py


In [28]:
import os

print(os.path.exists("W2D3_Webhook_Receiver.py"))
print(os.path.exists("store.db"))

True
True
